# Silver Layer: Data Transformation and Quality Checks

**Data Source:** S3 Bronze Delta (s3://sales-pipeline-data-lake-ratnajit/bronze/)  
**Destination:** S3 Silver Delta (s3://sales-pipeline-data-lake-ratnajit/silver/)

**Transformations:**
- Type casting: aht_minutes (STRING to INT), created_date (STRING to DATE)
- Trim whitespace from text columns
- Remove duplicate tickets (keep latest by ingestion timestamp)
- Data quality validation: null checks, duplicate counts

In [0]:
from pyspark.sql.functions import col, trim, to_date, row_number
from pyspark.sql.window import Window

# S3 paths - all data stored in S3
BRONZE_S3_PATH = 's3://sales-pipeline-data-lake-ratnajit/bronze/customer_support_tickets/'
SILVER_S3_PATH = 's3://sales-pipeline-data-lake-ratnajit/silver/customer_support_tickets/'

# Table names for Unity Catalog registration
BRONZE_TABLE = 'workspace.bronze.customer_support_tickets'
SILVER_TABLE = 'workspace.silver.customer_support_tickets'

print(f'Reading from S3: {BRONZE_S3_PATH}')
print(f'Writing to S3: {SILVER_S3_PATH}')

Reading from S3: s3://sales-pipeline-data-lake-ratnajit/bronze/customer_support_tickets/
Writing to S3: s3://sales-pipeline-data-lake-ratnajit/silver/customer_support_tickets/


In [0]:
# Read directly from S3 Delta
df_bronze = spark.read.format('delta').load(BRONZE_S3_PATH)

print(f'Bronze records: {df_bronze.count()}')
print(f'\nBronze schema:')
df_bronze.printSchema()

print('\nSample bronze data:')
df_bronze.show(5, truncate=False)

Bronze records: 1000

Bronze schema:
root
 |-- ticket_id: string (nullable = true)
 |-- agent_name: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- issue_type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- aht_minutes: string (nullable = true)
 |-- created_date: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_url: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- _page: integer (nullable = true)


Sample bronze data:
+---------+----------+-----------+----------+--------+-----------+------------+--------------------------+----------------------------------------------------------+------------------------------------+-----+
|ticket_id|agent_name|customer_id|issue_type|status  |aht_minutes|created_date|_ingested_at              |_source_url                                               |_batch_id                           |_page|
+---------+----------+-----------+----------+--------+--

In [0]:
# Type casting and trimming
df_cleaned = df_bronze.select(
    trim(col('ticket_id')).alias('ticket_id'),
    trim(col('agent_name')).alias('agent_name'),
    trim(col('customer_id')).alias('customer_id'),
    trim(col('issue_type')).alias('issue_type'),
    trim(col('status')).alias('status'),
    col('aht_minutes').cast('int').alias('aht_minutes'),
    to_date(col('created_date'), 'yyyy-MM-dd').alias('created_date'),
    col('_ingested_at'),
    col('_source_url'),
    col('_batch_id'),
    col('_page')
)

# Data quality check: nulls
null_ticket_id = df_cleaned.filter(col('ticket_id').isNull()).count()
null_aht = df_cleaned.filter(col('aht_minutes').isNull()).count()
null_date = df_cleaned.filter(col('created_date').isNull()).count()

print('Data Quality: Null Check')
print(f'Null ticket_id: {null_ticket_id}')
print(f'Null aht_minutes: {null_aht}')
print(f'Null created_date: {null_date}')

# Remove duplicates - keep latest record per ticket_id
window = Window.partitionBy('ticket_id').orderBy(col('_ingested_at').desc())
df_deduped = df_cleaned.withColumn('row_num', row_number().over(window)) \
    .filter(col('row_num') == 1) \
    .drop('row_num')

# Data quality check: duplicates removed
bronze_count = df_bronze.count()
silver_count = df_deduped.count()
duplicates_removed = bronze_count - silver_count

print(f'\nData Quality: Duplicate Check')
print(f'Bronze records: {bronze_count}')
print(f'Silver records: {silver_count}')
print(f'Duplicates removed: {duplicates_removed}')

print('\nSilver schema:')
df_deduped.printSchema()

Data Quality: Null Check
Null ticket_id: 0
Null aht_minutes: 0
Null created_date: 0

Data Quality: Duplicate Check
Bronze records: 1000
Silver records: 1000
Duplicates removed: 0

Silver schema:
root
 |-- ticket_id: string (nullable = true)
 |-- agent_name: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- issue_type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- aht_minutes: integer (nullable = true)
 |-- created_date: date (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_url: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- _page: integer (nullable = true)



In [0]:
# Create silver schema
spark.sql('CREATE SCHEMA IF NOT EXISTS workspace.silver')

# Write to S3 Delta - overwrite mode for clean snapshot
df_deduped.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .save(SILVER_S3_PATH)

# Register table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_TABLE}
USING DELTA
LOCATION '{SILVER_S3_PATH}'
""")

print(f'Silver table written: {df_deduped.count()} records')
print(f'Location: {SILVER_S3_PATH}')

Silver table written: 1000 records
Location: s3://sales-pipeline-data-lake-ratnajit/silver/customer_support_tickets/


In [0]:
# Verify silver table by reading directly from S3
df = spark.read.format('delta').load(SILVER_S3_PATH)

print(f'Total records: {df.count()}')
print(f'Distinct tickets: {df.select("ticket_id").distinct().count()}')

# Date range
from pyspark.sql.functions import min as _min, max as _max
date_stats = df.select(_min('created_date'), _max('created_date')).collect()[0]
print(f'Date range: {date_stats[0]} to {date_stats[1]}')

print('\nSample records:')
df.show(5, truncate=False)

# Verify type casting worked - aggregate by status
print('\nAverage AHT by status:')
df.groupBy('status').agg(
    {'aht_minutes': 'avg', 'ticket_id': 'count'}
).withColumnRenamed('avg(aht_minutes)', 'avg_aht') \
 .withColumnRenamed('count(ticket_id)', 'ticket_count') \
 .orderBy('status') \
 .show()

print('\nData quality summary:')
print(f'Null ticket_id: {df.filter(col("ticket_id").isNull()).count()}')
print(f'Null aht_minutes: {df.filter(col("aht_minutes").isNull()).count()}')
print(f'Null created_date: {df.filter(col("created_date").isNull()).count()}')

Total records: 1000
Distinct tickets: 1000
Date range: 2026-07-01 to 2026-09-29

Sample records:
+---------+----------+-----------+----------+--------+-----------+------------+--------------------------+----------------------------------------------------------+------------------------------------+-----+
|ticket_id|agent_name|customer_id|issue_type|status  |aht_minutes|created_date|_ingested_at              |_source_url                                               |_batch_id                           |_page|
+---------+----------+-----------+----------+--------+-----------+------------+--------------------------+----------------------------------------------------------+------------------------------------+-----+
|T001     |Rahul     |C101       |Billing   |Resolved|18         |2026-09-25  |2026-09-28 23:25:35.776108|idbfs:/2026-09-28/21/_5fcc1a59-e317-4a7f-bf23-6e81f6f8c2e1|be66c2a0-6d35-4c33-9cc3-5790f419ff72|1    |
|T002     |Priya     |C102       |Technical |Resolved|32         |2